In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f'{catalog_name}.{gold_schema}.dim_constructors'

**Step 1 - Read source tables**

- `silver.constructors`
- `gold.ref_nationality_region`

In [0]:
constructors_df = spark.read.table(f"{catalog_name}.{silver_schema}.constructors")
ref_nationality_region_df = spark.read.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

**Step 2 - Join `constructors` with `nationality_region_df` using `nationality`**

In [0]:
dim_constructors_df = (
            constructors_df
                .join(ref_nationality_region_df, 
                    constructors_df.nationality == ref_nationality_region_df.nationality,
                    "left")
                .select(
                    constructors_df.constructor_id,
                    constructors_df.constructor_name,
                    constructors_df.nationality,
                    ref_nationality_region_df.region.alias('nationality_region')
                )
)

**Step 3 - Write the transformed data to the `gold` `dim_constructors` table**

In [0]:
(
    dim_constructors_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(target_table)
)

In [0]:
%sql
select * from formula1.gold.dim_constructors